# Démo Neo4j — d'un récit d'incident à un graphe

**Contexte bancaire.** Chaque incident opérationnel est d'abord un *texte* :
> « Le 2025-10-14 à 00h24, l'équipe Sécurité Opérationnelle a détecté un incident
> critique sur le système « Passerelle SWIFT ». … L'équipe Supervision Paiements
> est intervenue et a rétabli le service le 2025-10-14 à 01h20. »

Pris un par un, ces récits ne se recoupent pas. **Transformés en graphe**, ils
révèlent des motifs : systèmes les plus fragiles, causes racines récurrentes,
équipes qui détectent vs qui résolvent, dépendances critiques.

Ce notebook déroule la démo de bout en bout :
**génération** → **NER** → **ontologie** → **graphe**.

## 0. Mise en place

On importe le référentiel métier et les deux étapes de la pipeline. Tout est
*offline* : aucune connexion, aucune clé API.

In [ ]:
import os, sys, json
sys.path.insert(0, os.getcwd())  # pour importer les scripts du dossier

import referentiel as ref
from generate_incidents import generer_incidents
from extract_entities import extraire, CHAMPS_EVALUES

print('Référentiel chargé :')
print(f"  {len(ref.EQUIPES)} équipes, {len(ref.SYSTEMES)} systèmes, "
      f"{len(ref.ROOT_CAUSES)} causes racines")

## 1. L'ontologie simplifiée

**Analogie.** Une ontologie, c'est le *plan de la ville* avant d'y placer les
habitants : on décide qu'il existe des *incidents*, des *équipes*, des *systèmes*
et des *causes racines*, et comment ils se relient. Le texte viendra ensuite
peupler ce plan.

```
(:Incident)-[:DETECTE_PAR]->(:Equipe)      (:Incident)-[:CAUSE_PAR]->(:RootCause)
(:Incident)-[:RESOLU_PAR]->(:Equipe)       (:Incident)-[:IMPACTE]->(:Systeme)
(:Equipe)-[:RESPONSABLE_DE]->(:Systeme)    (:Systeme)-[:DEPEND_DE]->(:Systeme)
```

In [ ]:
print('ÉQUIPES (:Equipe)')
for nom, dom in ref.EQUIPES.items():
    print(f'  - {nom:28s} [domaine: {dom}]')
print()
print('SYSTÈMES (:Systeme) — le contexte opérationnel')
for nom, meta in ref.SYSTEMES.items():
    print(f"  - {nom:26s} [type: {meta['type']}] resp: {meta['responsable']}")
print()
print('CAUSES RACINES (:RootCause)')
for lib, cat in ref.ROOT_CAUSES.items():
    print(f'  - {lib:44s} [{cat}]')

> **À retenir.** Quatre types de nœuds, six types de relations. La timeline
> (détection/résolution) est portée par des *propriétés* de l'incident — c'est le
> choix « simplifié » (détail dans `ontologie.md`).

## 2. Générer 100 récits d'incidents

On tire des combinaisons cohérentes (système, équipes, cause, sévérité, timeline)
et on les *raconte* avec plusieurs gabarits de phrases. On garde la **vérité
terrain** à côté du texte pour, plus tard, noter le NER.

In [ ]:
incidents = generer_incidents(nb=100, seed=42)
print(f'{len(incidents)} incidents générés.\n')
for inc in incidents[:3]:
    print(f"[{inc['id']}] {inc['titre']}")
    print('  ' + inc['description'])
    print()

## 3. NER — extraire les entités du texte

**Analogie.** Le NER, c'est un lecteur qui surligne dans le récit *qui* (équipes),
*quand* (timeline), *quoi* (système) et *pourquoi* (cause racine). Ici il ne voit
**que le texte** — jamais la vérité terrain.

In [ ]:
ex = incidents[0]
entites = extraire(ex['description'])
print('TEXTE :')
print(' ', ex['description'])
print('\nENTITÉS EXTRAITES :')
for k, v in entites.items():
    print(f'  {k:22s} : {v}')

Est-ce que ça marche vraiment ? On extrait sur les 100 incidents et on compare
champ par champ à la vérité terrain.

In [ ]:
compteurs = {c: 0 for c in CHAMPS_EVALUES}
for inc in incidents:
    e = extraire(inc['description'])
    gt = inc['ground_truth']
    for c in CHAMPS_EVALUES:
        if e.get(c) == gt.get(c):
            compteurs[c] += 1
n = len(incidents)
print('Taux de restitution vs vérité terrain :')
for c in CHAMPS_EVALUES:
    print(f'  {c:22s} : {compteurs[c]:3d}/{n}  ({100*compteurs[c]/n:5.1f} %)')
glob = sum(compteurs.values()); tot = n*len(CHAMPS_EVALUES)
print(f'  {"GLOBAL":22s} : {glob}/{tot}  ({100*glob/tot:5.1f} %)')

> **À retenir.** Comme on maîtrise la génération du texte, le NER par règles
> atteint ~100 %. Sur des incidents réels et bruités, on remplacerait cet
> extracteur par un modèle (spaCy, LLM) — **le reste de la pipeline ne change pas**.

## 4. Visualiser le graphe

Avant même de charger Neo4j, on peut voir la *colonne vertébrale* de l'ontologie
avec `networkx` : les systèmes, leurs dépendances (flèches), et l'équipe
responsable de chacun. `Core Banking` apparaît comme le **hub central**.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for s in ref.liste_systemes():
    G.add_node(s, kind='systeme')
for e in ref.liste_equipes():
    G.add_node(e, kind='equipe')
for a, b in ref.DEPENDANCES_SYSTEMES:
    G.add_edge(a, b, rel='DEPEND_DE')
for s, meta in ref.SYSTEMES.items():
    G.add_edge(meta['responsable'], s, rel='RESPONSABLE_DE')

# Positions manuelles (spring_layout instable pour un schéma pédagogique)
pos = {
    'Core Banking': (0.0, 0.0),
    'API Open Banking': (-1.6, 1.0), 'Virements SEPA': (1.6, 1.0),
    'Monétique Cartes': (1.8, -0.2), 'Passerelle SWIFT': (1.2, -1.3),
    'Batch de Réconciliation': (-0.2, -1.6),
    'Application Mobile': (-2.6, 1.9), 'Portail Web': (-2.7, 0.4),
    'Supervision Paiements': (2.9, 0.3), 'Run Core Banking': (0.0, 1.4),
    'Canaux Digitaux': (-3.4, 1.2), 'Data & Réconciliation': (-1.0, -2.6),
    'Infrastructure & Réseau': (0.9, -2.6), 'Sécurité Opérationnelle': (2.6, -1.8),
    'Monitoring 24/7 (NOC)': (3.4, -0.7),
}

fig, ax = plt.subplots(figsize=(13, 9))
sys_nodes = [n for n, d in G.nodes(data=True) if d['kind']=='systeme']
team_nodes = [n for n, d in G.nodes(data=True) if d['kind']=='equipe']
nx.draw_networkx_nodes(G, pos, nodelist=sys_nodes, node_color='#4C9BE8',
                       node_size=2600, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=team_nodes, node_color='#F2C14E',
                       node_shape='s', node_size=2200, ax=ax)
dep = [(u,v) for u,v,d in G.edges(data=True) if d['rel']=='DEPEND_DE']
resp = [(u,v) for u,v,d in G.edges(data=True) if d['rel']=='RESPONSABLE_DE']
nx.draw_networkx_edges(G, pos, edgelist=dep, edge_color='#2C3E50', width=2.0,
                       arrows=True, arrowsize=20, min_source_margin=25,
                       min_target_margin=25, connectionstyle='arc3,rad=0.1', ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=resp, edge_color='#C0392B', width=1.2,
                       style='dashed', arrows=True, arrowsize=14,
                       min_source_margin=22, min_target_margin=22,
                       connectionstyle='arc3,rad=0.12', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
ax.set_title('Ossature de l\'ontologie — systèmes (bleu), équipes (jaune)\n'
             'DEPEND_DE (trait plein) · RESPONSABLE_DE (pointillés rouges)',
             fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

Et l'agrégat qui intéresse le *run* : combien d'incidents par système et par
cause racine.

In [ ]:
from collections import Counter
par_systeme = Counter(inc['ground_truth']['systeme'] for inc in incidents)
par_cause = Counter(inc['ground_truth']['root_cause'] for inc in incidents)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
s_items = par_systeme.most_common()
ax1.barh([k for k,_ in s_items][::-1], [v for _,v in s_items][::-1], color='#4C9BE8')
ax1.set_title('Incidents par système impacté'); ax1.set_xlabel('nombre')
c_items = par_cause.most_common()
ax2.barh([k for k,_ in c_items][::-1], [v for _,v in c_items][::-1], color='#E67E22')
ax2.set_title('Incidents par cause racine'); ax2.set_xlabel('nombre')
plt.tight_layout()
plt.show()

## 5. Charger dans Neo4j et explorer

La pipeline complète écrit ces entités dans Neo4j :

```bash
docker compose up -d        # interface web sur http://localhost:7474
python3 load_neo4j.py       # charge le graphe via Bolt
```

Puis, dans **Neo4j Browser**, quelques requêtes de démo (voir `queries.cypher`) :

```cypher
// Top des causes racines
MATCH (i:Incident)-[:CAUSE_PAR]->(r:RootCause)
RETURN r.libelle AS cause, count(i) AS nb ORDER BY nb DESC;

// MTTR moyen par sévérité
MATCH (i:Incident)
RETURN i.severite, round(avg(i.mttr_minutes)) AS mttr_moyen_min;

// Détection → incident → système → résolution
MATCH p=(:Equipe)<-[:DETECTE_PAR]-(:Incident)-[:IMPACTE]->(:Systeme)
RETURN p LIMIT 25;
```

## À retenir

- Un **texte d'incident** contient, structurés implicitement : *qui, quand, quoi,
  pourquoi*. Le **NER** les rend explicites.
- Une **ontologie simplifiée** (4 nœuds, 6 relations) suffit à transformer 100
  récits isolés en un **graphe interrogeable**.
- Neo4j apporte l'**exploration visuelle** et les **requêtes de graphe** (chemins,
  analyse d'impact, corrélations cause ↔ système) impossibles à voir dans du texte.
- La pipeline est **modulaire** : remplacer l'extracteur par règles par un modèle
  n'impacte ni l'ontologie ni le chargement.